# RadCluster_2_1 — Digital-Twin Campaign Control

Run this notebook **on each machine**. It configures the run, verifies the
toolchain, launches the worker, and gives you a live view of where the
campaign stands against the experimental data.

| cell | what it does |
|---|---|
| **1 Configure** | every knob for this machine in one place |
| **2 Build check** | verifies the C++ solver; compiles it if missing *or stale* |
| **3 Agreement check** | confirms this machine reproduces the reference numbers |
| **4 Launch** | starts the worker in the background |
| **5 Monitor** | live progress, timing, remaining compute, results vs experiment |
| **6 Graceful stop** | halt without losing work |
| **7 Restart** | resume and keep everything already computed |
| **8 Tally** | merge all machines → Sobol indices |

**The one rule:** every machine must run the *same commit* and the *same
design file*. Cell 3 enforces it; cell 5 warns if it ever drifts.

## 1 — Configure

In [6]:
from pathlib import Path
import os, sys, json, subprocess

HERE = Path.cwd() if Path.cwd().name == 'digital_twin' else Path('RadCluster_2_1/digital_twin')
sys.path.insert(0, str(HERE))
import campaign_ops as ops

# ── THIS MACHINE ─────────────────────────────────────────────────────────
MACHINE     = 2        # 0,1,2,3 — MUST be unique across the four machines
N_MACHINES  = 4
WORKERS     = max(1, (os.cpu_count() or 4) - 2)   # single-threaded workers

# Each worker is ONE PROCESS running ONE THREAD (run_ensemble forces
# OMP_NUM_THREADS=1).  Many serial jobs beat few threaded ones here, and it
# also fixes the reduction order so a 24-core and an 8-core box give
# bit-identical answers for the same row — otherwise thread-count noise would
# show up in the Sobol indices as if it were parameter sensitivity.
#
# WEIGHTS: relative capacity of EVERY machine, in machine order — normally
# each one's WORKERS.  The design is then split in proportion to capacity
# instead of evenly.  With 6/14/22/22 workers an even split finishes at
# 31.7 h (three machines idle for the last 23); weighted, all four finish
# at ~12.0 h.  Set it to the SAME list on all four machines, or leave it
# None on all four for the plain even split.
WEIGHTS     = None     # e.g. [6, 14, 22, 22]

# ── THE RUN ──────────────────────────────────────────────────────────────
DESIGN      = HERE / 'design' / 'T2_design_v2.csv'
RESULTS     = HERE / 'results'
SOLVER_MODE = 'active_window'
                       # T0.2 measured 2026-08-02. active_window reproduces
                       # full_system to 4.1e-4 at the BOX CORNER (I=2400) and
                       # to printed precision at the nominal (I=1600), while
                       # being 2.4-3.2x faster with delta_FP 3 orders better.
                       # Preconditioner follows automatically (Jacobi here,
                       # Woodbury for full_system) -- never set it by hand.

I_GRID      = 3200     # MEASURED, not inferred. pile (content in the top 2%
                       # of the grid; >0.05 = the size readout is a ceiling
                       # artefact):
                       #     I     800     1600    2400    3200
                       #   nominal 0.4642  0.1474    --    0.0000
                       #   corner  0.5965  0.3608  0.1018  0.0037
                       # Both clear at 3200. d_100 then stops moving:
                       # 6.55 at I=3200 and 6.55 at I=6400 (nominal).
                       # 800 -- the value this notebook first shipped --
                       # fails even at the calibrated nominal.
                       # NOTE N_loops_100 is grid-INVARIANT to 0.4% across
                       # I=800..3200 and both box corners; only d_100_nm
                       # needs the grid. The two <100> observables have
                       # different requirements.

V_GRID      = 600      # Vacancy axis, added to the checks 2026-08-02 after
                       # it was found to be unguarded. Verified AT THE
                       # CAMPAIGN CONFIG: delta_FP=7.0e-12, mean_n_v=168.00,
                       # N_voids=9.119e20 -- clean. (The check_machine probe
                       # shipped V=120, where mean_n_v was 8x off and
                       # N_voids 10.6x off; that probe is now I=300/V=480.)
DOSE        = 0.1      # dpa, the Tier-2 LF dose
EQUATIONS   = 'discrete'   # 'bin_moment' once it is validated (much faster)
RTOL        = 1e-6
TIMEOUT_S   = 10800    # per row (3 h). NOT 3600: measured rows at I=3200
                       # take 1843-2484 s, but those ran with <=3 concurrent
                       # processes. At WORKERS=22 the same row is slower --
                       # memory-bandwidth contention, not CPU -- so a 1 h cap
                       # would time out a large fraction of them. A timeout is
                       # strictly worse than a slow row: it FAILS, and pairwise
                       # deletion then also discards its Saltelli partner.
LIMIT       = 0        # >0 = smoke test on the first N rows only

env = ops.environment()
print(f"machine   {env['machine_id']}  ({env['cpu_count']} cores, using {WORKERS} workers)")
print(f"branch    {env['branch']}  @ {env['git_sha'][:12]}" + ('  *** WORKTREE DIRTY ***' if env['worktree_dirty'] else ''))
print(f"design    {DESIGN.name}")
print(f"run       I={I_GRID} V={V_GRID} {DOSE} dpa {EQUATIONS} rtol={RTOL}")
print(f"solver    {SOLVER_MODE}  (preconditioner follows automatically)")
print(f"split     {'weighted ' + str(WEIGHTS) if WEIGHTS else 'even (row_id % M)'}")
if env['worktree_dirty']:
    print('\n  Uncommitted changes: this machine may not match the others.')
    print('  Commit or stash before a production run.')

machine   Mac.san.rr.com  (8 cores, using 6 workers)
branch    main  @ a9e512b9b8eb  *** WORKTREE DIRTY ***
design    T2_design_v2.csv
run       I=3200 V=600 0.1 dpa discrete rtol=1e-06
solver    active_window  (preconditioner follows automatically)
split     even (row_id % M)

  Uncommitted changes: this machine may not match the others.
  Commit or stash before a production run.


## 2 — Build check (auto-builds if needed)

The solver binary is **not** in git — each machine compiles its own. This also
catches a *stale* binary: if any `.cpp`/`.h`/`CMakeLists.txt` is newer than
`solver.exe`, it rebuilds. That is the case that silently produces results from
code you thought you had replaced.

In [7]:
info = ops.ensure_solver()          # ops.ensure_solver(force=True) to rebuild anyway
info

  solver: OK  /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/build/solver  sha 9366e57649416372


{'path': '/Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/build/solver',
 'exists': True,
 'sha256': '9366e57649416372',
 'mtime': 1785724403.5157971,
 'newest_source_mtime': 1785723402.5498254,
 'stale': False,
 'built': False}

## 3 — Machine agreement check

Runs a fixed ~30 s probe and compares 12 quantities against the committed
reference at `rtol=1e-6` — the campaign's own integration tolerance.

The bar was `1e-9` while every machine ran the same Windows/x86 build, where
CVODE is bit-deterministic. It cannot hold **across architectures**: macOS/arm64
(AppleClang + Accelerate) and Windows/x86 (MSVC + MKL) reassociate the same
reductions differently, which lands at ~1e-7 on the integrated fields while the
closed-form ones (`Di_eff`, `Dv_eff`, `conv_*`) still match to `0.0` exactly.
A gate tighter than the integrator's own accuracy rejects machines whose physics
is identical, so it now sits *at* that accuracy: above it is a real
discrepancy, below it is beneath the resolution the solver claims.

**Do not start a machine that fails this.** Its rows cannot be pooled with the
others, and the failure will not be visible in the physics.

In [8]:
r = subprocess.run([sys.executable, str(HERE / 'check_machine.py')],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print('AGREEMENT OK' if r.returncode == 0 else f'*** FAILED (rc={r.returncode}) — do not launch ***')

machine   Mac.san.rr.com  (macOS-15.7.4-arm64-arm-64bit)
python    3.9.6
git       a9e512b9b8eb
solver    9366e57649416372  exists=True
workbook  9253e7a0370af966

running probe (I=150, 0.02 dpa, ~30 s) ...

  reference generated on Nasr-Workstation (git 76efc2a46f6c, solver c28893e7d4a119e2)
  note: git SHA differs from the reference machine - pull first if that is not intentional.

  field                    this machine        reference    rel diff
  Di_eff                7.085348836e-12  7.085348836e-12    0.00e+00
  Dv_eff                2.125201773e-13  2.125201773e-13    0.00e+00
  conv_psuccess         2.181172484e-06  2.181172484e-06    0.00e+00
  conv_psuccess_abs     1.000000000e+00  1.000000000e+00    0.00e+00
  N_loops_100           1.679711672e+20  1.679711678e+20    3.62e-09
  N_loops_111           2.904994897e+23  2.904994914e+23    5.81e-09
  mean_n_100            2.016423182e+02  2.016423189e+02    3.00e-09
  mean_n_111            3.604016025e+01  3.604016025e+01    8

## 4 — Launch the worker

Starts in the background so the monitor cell stays usable. Safe to re-run:
rows already completed are skipped.

In [9]:
ops.clear_stop()          # make sure no stale STOP flag is present
RESULTS.mkdir(parents=True, exist_ok=True)
log = RESULTS / f'worker_machine{MACHINE}.log'
cmd = [sys.executable, '-u', str(HERE / 'run_ensemble.py'),
       '--design', str(DESIGN), '--machine', str(MACHINE), '--of', str(N_MACHINES),
       '--workers', str(WORKERS), '--I', str(I_GRID), '--V', str(V_GRID),
       ] + (['--weights', ','.join(str(w) for w in WEIGHTS)] if WEIGHTS else []) + [
       '--dose', str(DOSE), '--equations', EQUATIONS, '--rtol', str(RTOL),
       '--solver-mode', SOLVER_MODE,
       '--timeout-s', str(TIMEOUT_S)] + (['--limit', str(LIMIT)] if LIMIT else [])
print(' '.join(cmd))

# start_new_session=True puts the worker in its OWN session and process group.
# Without it the worker is in the KERNEL's process group, and Jupyter kills that
# group on shutdown -- so restarting the kernel, closing the notebook, or running
# this notebook headlessly silently killed a multi-hour campaign mid-row. The
# worker must outlive the notebook that started it; the STOP sentinel (cell 6),
# not process lifetime, is how it is meant to be halted.
proc = subprocess.Popen(cmd, stdout=open(log, 'a'), stderr=subprocess.STDOUT,
                        start_new_session=True)
print(f'\nlaunched pid {proc.pid}, logging to {log}')
print('detached: survives kernel restart -- stop it with cell 6, not by killing the kernel')

  no STOP flag set.
/Users/ghoni/Documents/GitHub/RadCluster/.EuroVenv/bin/python -u /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin/run_ensemble.py --design /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin/design/T2_design_v2.csv --machine 2 --of 4 --workers 6 --I 3200 --V 600 --dose 0.1 --equations discrete --rtol 1e-06 --solver-mode active_window --timeout-s 10800

launched pid 74345, logging to /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin/results/worker_machine2.log
detached: survives kernel restart -- stop it with cell 6, not by killing the kernel


## 5 — Live monitor

Refreshes in place. **Interrupt the kernel to stop watching — the campaign is
unaffected.** Shows coverage, per-machine progress, per-row timing, remaining
core-hours and ETA, why rows were rejected, provenance drift, and where the
ensemble sits against each experimental band.

On the observables table: a low *in band* fraction early on is normal — the
prior box is deliberately wide. Stuck at 0% after several hundred rows means
the box does not contain the data, which is itself a result.

In [10]:
ops.watch(DESIGN, RESULTS, n_machines=N_MACHINES,
          workers_per_machine=WORKERS, interval=60)

  refreshed 16:23:20  (every 60s; interrupt the kernel to stop watching)

CAMPAIGN  T2_design_v2.csv   p=21 N=16 conditions=N2,N5,I1
  progress  [#####################.........................]  46.7%  516/1104 rows
            admissible 21   inadmissible 495   failed 0   missing 588
            rejected because: {'grid-limited': 468, 'delta_FP>=1e-3': 439, 'dose-starved': 57}

  timing    per row  mean 58m31s   median 19m34s   p90 3h01m
            core-hours used 503.3   remaining ~573.5

   machine                 id  assigned   done   left     ETA @14w
         0     Mac.san.rr.com       276      1    275       19h09m
         1     Mac.san.rr.com       276    274      2        8m21s
         2     Mac.san.rr.com       276     94    182       12h40m
         3         MATRIX-PC2       276    147    129        8h59m

  *** PROVENANCE SPLIT on solver_sha256 - results are NOT comparable:
        unavailable  <- ['Mac.san.rr.com']
        9366e57649416372  <- ['Mac.san.rr.com']
      

{'design': '/Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin/design/T2_design_v2.csv',
 'meta': {'tier': 2,
  'N': 16,
  'p': 21,
  'seed': 20260801,
  'conditions': ['N2', 'N5', 'I1'],
  'rows_per_condition': 368,
  'rows_total': 1104,
  'parameters': ['eta',
   'f_cl_i',
   'f_cl_v',
   'E_m_i',
   'E_m_v',
   'E_m_h',
   'L_hat',
   'i_mobile',
   'v_mobile',
   'gamma_s',
   'E_b_v2',
   'lambda',
   'B_111',
   'E_b_i2',
   'E_b_hV_1',
   'Z_i',
   'Z_i_loop',
   'E_a0_conv',
   'dH2_conv',
   'phi_max_junc',
   'dH2_abs_conv'],
  'parameters_version': 'T2_v1',
  'revision_pending': [],
  'design_sha256': '9f212a959e477e86a529ccd314655159b8ec9980ea9936efd71bfebf73ccfc63',
  'git_sha': 'ba454f923634701bf7df79bb57acc7a5ffe7b914',
  'estimator': 'Saltelli; S_j pairs A_i with AB_i^(j), S_T_j pairs B_i with AB_i^(j)'},
 'total': 1104,
 'done': 516,
 'admissible': 21,
 'inadmissible': 495,
 'failed': 0,
 'missing': 588,
 'pct': 46.73913043478261,
 'wall_mean_s': 3511

In [ ]:
# One-shot snapshot instead of the live loop
st = ops.campaign_status(DESIGN, RESULTS, N_MACHINES, WORKERS)
ops.render_status(st, ops.load_targets())

### 5b — Inspect suspicious rows

If the monitor shows something unphysical, look before you stop. This lists the
worst offenders by category so you can tell a bad *parameter region* (a
result — record it) from a bad *model* (a reason to stop).

In [ ]:
import numpy as np
recs = list(ops.load_results(RESULTS).values())
bad  = [r for r in recs if not r.get('solver_rc') and not r.get('admissible')]
print(f'{len(bad)} inadmissible of {len(recs)}\n')
for r in sorted(bad, key=lambda z: -(z.get('pile_100') or 0))[:10]:
    print(f"  row {r['row_id']:6d} {r['condition']} pile100={r.get('pile_100')} "
          f"occ100={r.get('occ_100'):.3f} d100={r.get('d_100_nm'):.2f} "
          f"dose={r.get('dose_reached'):.3f} dFP={r.get('delta_FP'):.1e}")
print('\nfailed rows:')
for r in [x for x in recs if x.get('solver_rc')][:10]:
    print(f"  row {r['row_id']:6d}  {r.get('error','')[:110]}")

## 6 — Graceful stop

Writes a sentinel the worker checks between rows. It **stops submitting new
work, lets in-flight rows finish and be written, then exits cleanly**. Nothing
is lost and nothing partial is written.

Do *not* kill the kernel or the process — that discards every row currently
being computed (up to `WORKERS` of them, each possibly an hour of compute).

In [ ]:
ops.request_stop('unphysical d_100 seen in the monitor')   # put the real reason here
# then watch the tail of the log until it prints STOPPED:
print(open(log).read()[-1200:])

## 7 — Change something, then restart

Everything already computed is kept — `run_ensemble` skips `row_id`s already in
this machine's `.jsonl`.

**But resuming is only a benefit if the new rows are comparable to the old
ones.** If you change code, the solver, the workbook or the design, the runner
*refuses* to append (exit 3) and tells you which hash moved. Two honest options:

* **The change affects results** (a kernel, a parameter, the grid) — archive the
  old rows and start that machine's file clean. The elapsed compute is not
  wasted: it is preserved as a labelled prior campaign.
* **The change cannot affect results** (a comment, the README) — re-run with
  `--allow-mixed`.

Changing `I_GRID`, `DOSE` or `EQUATIONS` **always** invalidates prior rows —
they are not part of `θ`, so nothing else records that they moved.

In [ ]:
# Archive this machine's rows under a label, keeping them for the record
import shutil, time as _t
src = RESULTS / f'{DESIGN.stem}_machine{MACHINE}.jsonl'
if src.exists():
    tag = input('label for the archived campaign (e.g. pre-Ea0-fix): ').strip() or _t.strftime('%Y%m%d_%H%M')
    dst = RESULTS / 'archive' / f'{src.stem}__{tag}.jsonl'
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f'archived -> {dst}')
else:
    print('nothing to archive')

In [ ]:
ops.clear_stop()
# re-run cell 2 (rebuild if you touched C++), then cell 3, then cell 4.

## 8 — Provisional twin map (results tables)

Joins the design (θ) with the results (observables) and the experimental
targets into one named artifact set in `report/`:

| file | contents |
|---|---|
| `provisional_twin_map.md` | the report |
| `..._runs.csv` | one row per evaluation: θ + observables + admissibility + provenance |
| `..._runs_admissible.csv` | usable rows only |
| `..._observables.csv` | model p05/median/p95 vs the experimental band, per condition |
| `..._targets.csv` | the experimental table |

*Provisional* is literal: it contains **no extrapolation to experimental
conditions** and **no posterior UQ**. Both require the Tier-4 emulator. The
spread shown is prior-predictive, not a posterior.

In [ ]:
r = subprocess.run([sys.executable, str(HERE / 'make_tables.py'),
                    '--design', str(DESIGN), '--results', str(RESULTS),
                    '--out', str(HERE / 'report')],
                   capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:])

from IPython.display import Markdown, display
m = HERE / 'report' / 'provisional_twin_map.md'
if m.exists():
    display(Markdown(m.read_text(encoding='utf-8')))

## 9 — Tally across machines

Each machine writes its own file, so they never conflict; `git pull` (or a
shared drive) brings them together. Merging is keyed on `row_id`, so it is
order-independent and idempotent — safe to run on a partial campaign and again
later.

In [ ]:
r = subprocess.run([sys.executable, str(HERE / 'merge_and_sobol.py'),
                    '--design', str(DESIGN), '--results', str(RESULTS),
                    '--out', str(HERE / 'report')],
                   capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-2000:])